# Part 2: Spark Structured Streaming Prediction Pipeline

This notebook consumes weather events from Kafka, validates and aggregates them to the model serving grain, applies the saved A2A pipeline model, persists prediction outputs, and republishes aggregate streams to Kafka.

## 2.1 Spark Session Configuration

In [1]:
import os
from pyspark import SparkContext
from pyspark.sql import SparkSession

#Setup enviroment for Streaming
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.5.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 pyspark-shell'

In [2]:
#Starting Spark Session
APP_NAME = "A2B-Streaming-Prediction"
TIMEZONE = "UTC"
DEFAULT_CHECKPOINT_DIR = "checkpoints"
LOCAL_SHUFFLE_PARTITIONS = "8"


def reset_spark_gateway_if_needed():
    """Clear stale PySpark state left behind by spark.stop() or a crashed JVM."""
    existing_spark = globals().get("spark")
    if existing_spark is not None:
        try:
            for query in existing_spark.streams.active:
                query.stop()
            existing_spark.stop()
            print("Stopped existing Spark session before creating a fresh one.")
        except Exception as exc:
            print(f"Clearing stale Spark session state after {type(exc).__name__}: {exc}")

    SparkSession._instantiatedSession = None
    SparkSession._activeSession = None
    SparkContext._active_spark_context = None
    SparkContext._gateway = None
    SparkContext._jvm = None


reset_spark_gateway_if_needed()

spark = (
    SparkSession.builder
        .appName(APP_NAME)
        .master("local[4]")
        .config("spark.sql.session.timeZone", TIMEZONE)
        .config("spark.sql.shuffle.partitions", LOCAL_SHUFFLE_PARTITIONS)
        .config("spark.sql.streaming.checkpointLocation", DEFAULT_CHECKPOINT_DIR)
        .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark timezone: {spark.conf.get('spark.sql.session.timeZone')}")
print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

Spark timezone: UTC
Shuffle partitions: 8


## 2.2 Static and Streaming Schemas

In [3]:
from pyspark.sql.types import ( StructType, StructField, IntegerType, StringType, TimestampType, DoubleType, DecimalType )

#manually defining schemas for each file

#PURE STRING SCHEMA FOR INITIAL READING FROM QUERY
buildings_schema = StructType([
    StructField("site_id", StringType(), False),
    StructField("building_id", StringType(), False),
    StructField("primary_use", StringType(), True),
    StructField("square_feet", IntegerType(), True),
    StructField("floor_count", IntegerType(), True),
    StructField("row_id", StringType(), False),
    StructField("year_built", IntegerType(), True),
    StructField("latent_y", IntegerType(), True),
    StructField("latent_s", DoubleType(), True),
    StructField("latent_r", IntegerType(), True)
])

# LOAD BUILDINGS (STATIC SET)
buildings = (
    spark.read
    .option("header", True)
    .schema(buildings_schema)
    .csv("new_building_information.csv")
    .dropDuplicates(["building_id"])
    .cache()  # keep in memory (uses less memory later when joining)
)

#SCHEMA FOR WEATHER DATA AS DEFINED IN A2A
weather_schema = StructType([
    StructField("month", DoubleType(), True),
    StructField("site_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("air_temperature", DoubleType(), True),
    StructField("cloud_coverage", DoubleType(), True), #If read as Int, will read each row as null, so read as Double
    StructField("dew_temperature", DoubleType(), True),
    StructField("sea_level_pressure", DoubleType(), True),
    StructField("wind_direction", DoubleType(), True), #If read as Int, will read each row as null, so read as Double
    StructField("wind_speed", DoubleType(), True),
    StructField("weather_ts", IntegerType(), True),
    StructField("season_peak", DoubleType(), True)
])


## 2.3 Kafka Weather Stream Ingestion

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

KAFKA_SERVERS = "kafka:9092"
TOPIC = "weather_stream"

# Live demo mode: start Task 2 before Task 1 and consume newly produced messages.
# For backlog replay after deleting checkpoints, change this to "earliest" before starting the stream.
STARTING_OFFSETS = "latest"
MAX_OFFSETS_PER_TRIGGER = 2000

# Incoming payload: everything StringType except weather_ts IntType.
# Parse as strings first, then cast explicitly so malformed values can be filtered.
weather_in_schema = StructType([
    StructField("month", StringType(), True),
    StructField("site_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("air_temperature", StringType(), True),
    StructField("cloud_coverage", StringType(), True),
    StructField("dew_temperature", StringType(), True),
    StructField("sea_level_pressure", StringType(), True),
    StructField("wind_direction", StringType(), True),
    StructField("wind_speed", StringType(), True),
    StructField("weather_ts", IntegerType(), True),
    StructField("season_peak", StringType(), True)
])

weather_stream = (
    spark.readStream.format("kafka")
         .option("kafka.bootstrap.servers", KAFKA_SERVERS)
         .option("subscribe", TOPIC)
         .option("startingOffsets", STARTING_OFFSETS)
         .option("maxOffsetsPerTrigger", MAX_OFFSETS_PER_TRIGGER)
         .load()
)

weather_json = weather_stream.select(
    F.col("key").cast("string").alias("kafka_key"),
    F.col("value").cast("string").alias("value"),
    F.col("timestamp").alias("kafka_ts")
)

parsed = (
    weather_json
      .select("kafka_key", "kafka_ts", F.from_json("value", weather_in_schema).alias("j"))
      .select("kafka_key", "kafka_ts", "j.*")
)

weather_typed = parsed.select(
    F.col("month").cast("int").alias("month"),
    F.col("site_id").cast("string").alias("site_id"),
    F.to_timestamp("timestamp").alias("timestamp"),
    F.col("air_temperature").cast("double").alias("air_temperature"),
    F.col("cloud_coverage").cast("double").alias("cloud_coverage"),
    F.col("dew_temperature").cast("double").alias("dew_temperature"),
    F.col("sea_level_pressure").cast("double").alias("sea_level_pressure"),
    F.col("wind_direction").cast("double").alias("wind_direction"),
    F.col("wind_speed").cast("double").alias("wind_speed"),
    F.col("weather_ts").cast("long").alias("weather_ts"),
    F.col("season_peak").cast("double").alias("season_peak"),
    "kafka_key",
    "kafka_ts"
)

In [5]:
weather_typed.printSchema()

root
 |-- month: integer (nullable = true)
 |-- site_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- air_temperature: double (nullable = true)
 |-- cloud_coverage: double (nullable = true)
 |-- dew_temperature: double (nullable = true)
 |-- sea_level_pressure: double (nullable = true)
 |-- wind_direction: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- weather_ts: long (nullable = true)
 |-- season_peak: double (nullable = true)
 |-- kafka_key: string (nullable = true)
 |-- kafka_ts: timestamp (nullable = true)



## 2.4 Event-Time Watermarking and Deduplication

In [6]:
from pyspark.sql import functions as F

required_weather_cols = [
    "site_id", "timestamp", "weather_ts", "air_temperature", "cloud_coverage",
    "dew_temperature", "sea_level_pressure", "wind_direction", "wind_speed", "season_peak"
]

# Drop malformed critical rows before stateful processing. Do not silently turn bad values into zeros.
weather_valid = weather_typed.na.drop(subset=required_weather_cols)

# weather_ts is the producer's arrival/event timestamp used for lateness. Business date/hour_block
# are derived after this filter, so all downstream aggregations consume data that passed the same
# event-time rule.
weather_with_wmk = (
    weather_valid
      .withColumn("event_time", F.to_timestamp(F.from_unixtime(F.col("weather_ts"))))
      .na.drop(subset=["event_time"])
      .withWatermark("event_time", "5 seconds")
)

weather_clean = weather_with_wmk.dropDuplicates(["site_id", "timestamp"])


## 2.5 Block-Level Feature Engineering

In [ ]:
# Row-level serving preparation shared by the consolidated q7 foreachBatch stream.
# The model is still scored at building/date/hour_block grain, but the 6-hour
# aggregation is now built inside foreachBatch from compacted hourly weather state.
from pyspark.sql.functions import broadcast

imp_cols = ["air_temperature", "cloud_coverage", "dew_temperature",
            "sea_level_pressure", "wind_direction", "wind_speed"]
model_weather_cols = ["air_temperature", "cloud_coverage", "dew_temperature"]
REQUIRED_HOUR_BLOCKS = ["00-06", "06-12", "12-18", "18-24"]

weather_prepared = (
    weather_clean
       .withColumn("date", F.to_date("timestamp"))
       .withColumn("hour", F.hour("timestamp"))
       .withColumn("hour_block",
           F.when(F.col("hour") <= 5, F.lit("00-06"))
            .when(F.col("hour") <= 11, F.lit("06-12"))
            .when(F.col("hour") <= 17, F.lit("12-18"))
            .otherwise(F.lit("18-24"))
       )
       .withColumn("timestamp_key", F.date_format("timestamp", "yyyy-MM-dd HH:mm:ss"))
       .withColumn("weather_record_key", F.concat_ws("|", F.col("site_id"), F.col("timestamp_key")))
       .drop("timestamp_key")
       .filter(F.col("hour_block").isin(REQUIRED_HOUR_BLOCKS))
)

cat_features = ["hour_block", "primary_use", "site_id"]
num_features = ["latent_y", "latent_s", "latent_r", "avg_air_temperature_6h",
                "avg_cloud_coverage_6h", "avg_dew_temperature_6h",
                "season_peak", "is_weekend", "floor_count"]

site_building_counts = (
    buildings
      .groupBy("site_id")
      .agg(F.countDistinct("building_id").cast("long").alias("site_building_count"))
      .cache()
)


## 2.6 Model Loading and Streaming Predictions

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml import PipelineModel

# Loading Pipeline model; it handles StringIndexer, OHE, assembler, and regressor stages.
pipe = PipelineModel.load("../A2A/models/gbt_best_model")


def build_serving_features_from_weather_blocks(weather_blocks_df):
    feature_df = weather_blocks_df.join(broadcast(buildings), on="site_id", how="inner")

    feature_df = feature_df.withColumn(
        "is_weekend",
        F.when(F.dayofweek("date").isin([1, 7]), F.lit(1)).otherwise(F.lit(0))
    )

    for c in num_features:
        feature_df = feature_df.withColumn(c, F.col(c).cast("double"))

    feature_df = feature_df.na.drop(
        subset=["building_id", "site_id", "date", "hour_block", "primary_use"] + num_features
    )

    return (
        feature_df
          .withColumn("block_start_time",
              F.when(F.col("hour_block") == "00-06", F.lit("00:00:00"))
               .when(F.col("hour_block") == "06-12", F.lit("06:00:00"))
               .when(F.col("hour_block") == "12-18", F.lit("12:00:00"))
               .otherwise(F.lit("18:00:00"))
          )
          .withColumn("wdw_start", F.to_timestamp(F.concat_ws(" ", F.col("date").cast("string"), F.col("block_start_time"))))
          .withColumn("wdw_end", F.expr("wdw_start + INTERVAL 6 HOURS"))
          .withColumn("date_str", F.date_format("date", "yyyy-MM-dd"))
          .withColumn("record_key", F.concat_ws("|", F.col("building_id"), F.col("date_str"), F.col("hour_block")))
    )


def score_weather_blocks(weather_blocks_df):
    serving_df = build_serving_features_from_weather_blocks(weather_blocks_df)
    return (
        pipe.transform(serving_df)
            .withColumn("prediction", F.when(F.col("prediction") < 0, F.lit(0.0)).otherwise(F.col("prediction")))
            .drop("block_start_time", "date_str")
    )


In [ ]:
# Notebook-visible Query 6 monitor
# q6a/q6b/q6c samples are captured by the consolidated q7 foreachBatch writer below.
# This avoids running duplicate streaming queries over the same Kafka -> model path.
from IPython.display import clear_output, display
from datetime import datetime, timezone
import time
import pandas as pd

STREAM_MONITOR = {
    "q6a": {"description": "Block-level building predictions", "batch_id": None, "rows": 0, "sample": [], "updated_at": None},
    "q6b": {"description": "Building 6-hour predictions", "batch_id": None, "rows": 0, "sample": [], "updated_at": None},
    "q6c": {"description": "Site daily predictions", "batch_id": None, "rows": 0, "sample": [], "updated_at": None},
}


def _utc_now_str():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")


def update_monitor_from_batch(name, batch_df, batch_id, limit=10):
    row_count = batch_df.count()
    sample_rows = []
    if row_count:
        sample_rows = [row.asDict(recursive=True) for row in batch_df.limit(limit).collect()]

    STREAM_MONITOR[name].update({
        "batch_id": batch_id,
        "rows": row_count,
        "sample": sample_rows,
        "updated_at": _utc_now_str(),
    })
    return row_count


def show_stream_monitor():
    active_rows = []
    try:
        active_queries = list(spark.streams.active)
    except Exception as exc:
        print("Spark session is not reachable from this notebook kernel.")
        print("Restart the kernel, rerun the Spark setup cells from the top, then start q7_prediction_outputs before using the monitor.")
        print(f"Spark error: {type(exc).__name__}: {exc}")
        return

    for q in active_queries:
        progress = q.lastProgress or {}
        active_rows.append({
            "name": q.name,
            "active": q.isActive,
            "batch_id": progress.get("batchId"),
            "input_rows": progress.get("numInputRows"),
            "processed_rows_per_second": progress.get("processedRowsPerSecond"),
            "exception": str(q.exception()) if q.exception() else None,
        })

    print("Active streaming queries")
    display(pd.DataFrame(active_rows) if active_rows else pd.DataFrame(columns=["name", "active", "batch_id", "input_rows", "processed_rows_per_second", "exception"]))

    summary_rows = []
    for name, state in STREAM_MONITOR.items():
        summary_rows.append({
            "query": name,
            "description": state["description"],
            "last_batch_id": state["batch_id"],
            "last_batch_rows": state["rows"],
            "sample_rows": len(state["sample"]),
            "updated_at": state["updated_at"],
        })

    print("\nQuery 6 sample capture")
    display(pd.DataFrame(summary_rows))

    for name, state in STREAM_MONITOR.items():
        print(f"\n{name}: {state['description']}")
        sample = state.get("sample") or []
        display(pd.DataFrame(sample) if sample else pd.DataFrame())


def watch_stream_monitor(seconds=60, interval=2):
    deadline = time.time() + seconds
    while time.time() < deadline:
        clear_output(wait=True)
        show_stream_monitor()
        time.sleep(interval)


In [ ]:
# q6a no longer has a separate console stream.
# Start q7_prediction_outputs, then run show_stream_monitor() or watch_stream_monitor().
print("q6a samples are captured by q7_prediction_outputs.")


In [ ]:
# Live Stream Monitor
# Run this cell repeatedly, or call watch_stream_monitor(seconds=60, interval=2) for automatic refresh.
show_stream_monitor()

In [ ]:
# Building-level 6-hour prediction rows are produced inside q7_prediction_outputs.
# The model already predicts a 6-hour building total, so the output uses the model prediction directly.
print("q6b samples are captured by q7_prediction_outputs.")


In [ ]:
# q6b no longer has a separate foreachBatch/show stream.
# Start q7_prediction_outputs, then run show_stream_monitor() or watch_stream_monitor().
print("q6b samples are captured by q7_prediction_outputs.")


In [ ]:
# Site-level daily prediction builder.
# Daily rows are rebuilt from compacted q7b building/date/hour_block state, not from only the current micro-batch.
def build_site_daily_from_building_blocks(building_blocks_df):
    required_blocks_col = F.array(*[F.lit(b) for b in REQUIRED_HOUR_BLOCKS])

    daily_df = (
        building_blocks_df
          .filter(F.col("hour_block").isin(REQUIRED_HOUR_BLOCKS))
          .groupBy("site_id", "date")
          .agg(
              F.sum("energy_consumption_6h").alias("daily_consumption_site"),
              F.max("event_time").alias("event_time"),
              F.min("wdw_start").alias("wdw_start"),
              F.max("wdw_end").alias("wdw_end"),
              F.count("*").cast("long").alias("building_block_count"),
              F.countDistinct("hour_block").cast("long").alias("hour_block_count"),
              F.array_sort(F.collect_set("hour_block")).alias("observed_hour_blocks")
          )
          .withColumn("missing_hour_blocks", F.array_except(required_blocks_col, F.col("observed_hour_blocks")))
          .withColumn("has_all_hour_blocks", F.size("missing_hour_blocks") == F.lit(0))
          .join(site_building_counts, on="site_id", how="left")
          .withColumn("expected_block_count", (F.col("site_building_count") * F.lit(4)).cast("long"))
          .withColumn(
              "is_complete_day",
              F.col("has_all_hour_blocks") & (F.col("building_block_count") >= F.col("expected_block_count"))
          )
          .withColumn("date_str", F.date_format("date", "yyyy-MM-dd"))
          .withColumn("record_key", F.concat_ws("|", F.col("site_id"), F.col("date_str")))
          .drop("date_str")
    )

    return daily_df

print("q6c samples are captured by q7_prediction_outputs.")


In [ ]:
# q6c no longer has a separate foreachBatch/show stream.
# Start q7_prediction_outputs, then run show_stream_monitor() or watch_stream_monitor().
print("q6c samples are captured by q7_prediction_outputs.")


## 2.7 Persisting Prediction Streams to Parquet

In [ ]:
# Consolidated q7 production stream.
# q7a, q7b, q7c, and the Kafka outboxes are derived from the same micro-batch.
# Current-state tables use versioned partition folders plus one small manifest commit,
# so a failed batch cannot expose half-rewritten snapshots.
import os
import re
import json
import shutil
import uuid
from pyspark.sql import Window
from pyspark.sql.types import ArrayType, BooleanType, DateType, LongType

BASE_A2B_PARQUET = "parquet/a2b"
MANIFEST_PATH = f"{BASE_A2B_PARQUET}/_manifest/current.json"
VERSION_ROOT = f"{BASE_A2B_PARQUET}/_versions"
OUTBOX_STAGE_ROOT = f"{BASE_A2B_PARQUET}/_outbox_staging"
P7A_OUTBOX = f"{BASE_A2B_PARQUET}/_kafka_outbox/7a_predictions"
P7B_OUTBOX = f"{BASE_A2B_PARQUET}/_kafka_outbox/7b_building_6h"
P7C_OUTBOX = f"{BASE_A2B_PARQUET}/_kafka_outbox/7c_site_daily"

WEATHER_TABLE = "weather_hourly"
Q7A_TABLE = "7a_predictions"
Q7B_TABLE = "7b_building_6h"
Q7C_TABLE = "7c_site_daily"

# Backwards-compatible logical names used by optional diagnostics. These are manifest-backed tables,
# not folders that should be read directly with spark.read.parquet().
P7A_SNAPSHOT = Q7A_TABLE
P7B_SNAPSHOT = Q7B_TABLE
P7C_SNAPSHOT = Q7C_TABLE

weather_state_cols = [
    "weather_record_key", "month", "site_id", "timestamp", "air_temperature",
    "cloud_coverage", "dew_temperature", "sea_level_pressure", "wind_direction",
    "wind_speed", "weather_ts", "season_peak", "event_time", "date", "hour", "hour_block"
]

schema_weather_state = StructType([
    StructField("weather_record_key", StringType(), True),
    StructField("month", IntegerType(), True),
    StructField("site_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("air_temperature", DoubleType(), True),
    StructField("cloud_coverage", DoubleType(), True),
    StructField("dew_temperature", DoubleType(), True),
    StructField("sea_level_pressure", DoubleType(), True),
    StructField("wind_direction", DoubleType(), True),
    StructField("wind_speed", DoubleType(), True),
    StructField("weather_ts", LongType(), True),
    StructField("season_peak", DoubleType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("date", DateType(), True),
    StructField("hour", IntegerType(), True),
    StructField("hour_block", StringType(), True),
])

schema_7a_files = StructType([
    StructField("record_key", StringType(), True),
    StructField("hour_block", StringType(), True),
    StructField("prediction", DoubleType(), True),
    StructField("date", DateType(), True),
    StructField("site_id", StringType(), True),
    StructField("building_id", StringType(), True),
    StructField("wdw_start", TimestampType(), True),
    StructField("wdw_end", TimestampType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("weather_rows_in_block", LongType(), True),
])

schema_7b_stream = StructType([
    StructField("record_key", StringType(), True),
    StructField("wdw_start", TimestampType(), True),
    StructField("wdw_end", TimestampType(), True),
    StructField("date", DateType(), True),
    StructField("hour_block", StringType(), True),
    StructField("building_id", StringType(), True),
    StructField("site_id", StringType(), True),
    StructField("energy_consumption_6h", DoubleType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("weather_rows_in_block", LongType(), True),
])

schema_7c_files = StructType([
    StructField("site_id", StringType(), True),
    StructField("date", DateType(), True),
    StructField("daily_consumption_site", DoubleType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("wdw_start", TimestampType(), True),
    StructField("wdw_end", TimestampType(), True),
    StructField("building_block_count", LongType(), True),
    StructField("hour_block_count", LongType(), True),
    StructField("observed_hour_blocks", ArrayType(StringType()), True),
    StructField("missing_hour_blocks", ArrayType(StringType()), True),
    StructField("has_all_hour_blocks", BooleanType(), True),
    StructField("site_building_count", LongType(), True),
    StructField("expected_block_count", LongType(), True),
    StructField("is_complete_day", BooleanType(), True),
    StructField("record_key", StringType(), True),
])

cols_7a = ["record_key", "hour_block", "prediction", "date", "site_id", "building_id", "wdw_start", "wdw_end", "event_time", "weather_rows_in_block"]
cols_7b = ["record_key", "wdw_start", "wdw_end", "date", "hour_block", "building_id", "site_id", "energy_consumption_6h", "event_time", "weather_rows_in_block"]
cols_7c = ["site_id", "date", "daily_consumption_site", "event_time", "wdw_start", "wdw_end", "building_block_count", "hour_block_count", "observed_hour_blocks", "missing_hour_blocks", "has_all_hour_blocks", "site_building_count", "expected_block_count", "is_complete_day", "record_key"]


def ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)


def _empty_manifest():
    return {"version": 0, "last_batch_id": None, "updated_at": None, "tables": {}}


def load_snapshot_manifest():
    if not os.path.exists(MANIFEST_PATH):
        return _empty_manifest()
    try:
        with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
            manifest = json.load(f)
    except Exception:
        return _empty_manifest()
    manifest.setdefault("version", 0)
    manifest.setdefault("tables", {})
    return manifest


def write_snapshot_manifest(manifest):
    ensure_parent_dir(MANIFEST_PATH)
    tmp_path = f"{MANIFEST_PATH}.tmp_{uuid.uuid4().hex}"
    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, sort_keys=True)
    os.replace(tmp_path, MANIFEST_PATH)


def _partition_value(value):
    if value is None:
        return "__null__"
    if hasattr(value, "isoformat"):
        return value.isoformat()
    return str(value)


def partition_key_from_values(values):
    return "|".join(_partition_value(v) for v in values)


def safe_partition_name(key):
    return re.sub(r"[^A-Za-z0-9_.=-]+", "_", key)


def collect_partition_keys(df, partition_cols):
    return [tuple(row[c] for c in partition_cols) for row in df.select(*partition_cols).dropDuplicates().collect()]


def filter_partition(df, partition_cols, values):
    condition = F.lit(True)
    for col_name, value in zip(partition_cols, values):
        if value is None:
            condition = condition & F.col(col_name).isNull()
        else:
            condition = condition & (F.col(col_name) == F.lit(value))
    return df.filter(condition)


def read_active_table(table_name, schema, partition_keys=None):
    manifest = load_snapshot_manifest()
    table_paths = manifest.get("tables", {}).get(table_name, {})

    if partition_keys is None:
        paths = sorted(set(table_paths.values()))
    else:
        paths = []
        for key_values in partition_keys:
            key = partition_key_from_values(key_values)
            path = table_paths.get(key)
            if path:
                paths.append(path)
        paths = sorted(set(paths))

    paths = [p for p in paths if os.path.exists(p)]
    if not paths:
        return spark.createDataFrame([], schema)
    return spark.read.schema(schema).parquet(*paths)


def write_partition_versions(df, table_name, partition_cols, partition_keys, batch_id):
    written = {}
    if not partition_keys:
        return written

    version_id = f"batch_{int(batch_id)}_{uuid.uuid4().hex}"
    table_version_root = f"{VERSION_ROOT}/{table_name}/{version_id}"

    for key_values in partition_keys:
        part_df = filter_partition(df, partition_cols, key_values)
        if part_df.limit(1).count() == 0:
            continue
        key = partition_key_from_values(key_values)
        part_path = f"{table_version_root}/{safe_partition_name(key)}"
        part_df.coalesce(1).write.mode("errorifexists").parquet(part_path)
        written[key] = part_path

    return written


def commit_manifest_updates(table_updates, batch_id):
    manifest = load_snapshot_manifest()
    new_manifest = json.loads(json.dumps(manifest))
    new_manifest["version"] = int(new_manifest.get("version", 0)) + 1
    new_manifest["last_batch_id"] = int(batch_id)
    new_manifest["updated_at"] = _utc_now_str()
    new_manifest.setdefault("tables", {})

    for table_name, updates in table_updates.items():
        if not updates:
            continue
        new_manifest["tables"].setdefault(table_name, {})
        new_manifest["tables"][table_name].update(updates)

    write_snapshot_manifest(new_manifest)


def stage_parquet_if_rows(df, table_name, batch_id):
    if df.limit(1).count() == 0:
        return None
    stage_path = f"{OUTBOX_STAGE_ROOT}/batch_{int(batch_id)}_{uuid.uuid4().hex}/{table_name}"
    df.coalesce(1).write.mode("errorifexists").parquet(stage_path)
    return stage_path


def publish_staged_parquet_files(stage_path, outbox_root):
    if not stage_path:
        return
    os.makedirs(outbox_root, exist_ok=True)
    for root, _, files in os.walk(stage_path):
        for file_name in files:
            if file_name.endswith(".parquet"):
                src = os.path.join(root, file_name)
                dst = os.path.join(outbox_root, f"{uuid.uuid4().hex}_{file_name}")
                shutil.move(src, dst)
    shutil.rmtree(stage_path, ignore_errors=True)


def dedupe_latest(df, key_cols, order_cols):
    order_exprs = [F.col(c).desc_nulls_last() for c in order_cols]
    window = Window.partitionBy(*key_cols).orderBy(*order_exprs)
    return (
        df.withColumn("_rn", F.row_number().over(window))
          .filter(F.col("_rn") == 1)
          .drop("_rn")
    )


def build_weather_blocks_from_state(weather_state_df, touched_blocks_df):
    return (
        weather_state_df
          .join(touched_blocks_df, on=["site_id", "date", "hour_block"], how="inner")
          .groupBy("site_id", "date", "hour_block")
          .agg(
              *[F.avg(c).alias(f"avg_{c}_6h") for c in imp_cols],
              F.max("season_peak").alias("season_peak"),
              F.max("event_time").alias("event_time"),
              F.countDistinct("timestamp").cast("long").alias("weather_rows_in_block")
          )
          .filter(F.col("weather_rows_in_block") >= 6)
    )


def empty_df(schema):
    return spark.createDataFrame([], schema)


def write_prediction_outputs(batch_df, batch_id):
    weather_batch = (
        batch_df
          .select(*weather_state_cols)
          .dropDuplicates(["weather_record_key"])
          .cache()
    )

    try:
        if weather_batch.limit(1).count() == 0:
            update_monitor_from_batch("q6a", empty_df(schema_7a_files), batch_id)
            update_monitor_from_batch("q6b", empty_df(schema_7b_stream), batch_id)
            update_monitor_from_batch("q6c", empty_df(schema_7c_files), batch_id)
            return

        touched_weather_keys = collect_partition_keys(weather_batch, ["site_id", "date", "hour_block"])
        existing_weather = read_active_table(WEATHER_TABLE, schema_weather_state, touched_weather_keys)
        weather_state = dedupe_latest(
            existing_weather.unionByName(weather_batch.select(*weather_state_cols), allowMissingColumns=True),
            ["weather_record_key"],
            ["event_time", "weather_ts"]
        ).select(*weather_state_cols).cache()

        q7a_stage = q7b_stage = q7c_stage = None
        try:
            weather_updates = write_partition_versions(weather_state, WEATHER_TABLE, ["site_id", "date", "hour_block"], touched_weather_keys, batch_id)
            touched_blocks = weather_batch.select("site_id", "date", "hour_block").dropDuplicates()
            weather_blocks = build_weather_blocks_from_state(weather_state, touched_blocks).cache()

            try:
                if weather_blocks.limit(1).count() == 0:
                    commit_manifest_updates({WEATHER_TABLE: weather_updates}, batch_id)
                    update_monitor_from_batch("q6a", empty_df(schema_7a_files), batch_id)
                    update_monitor_from_batch("q6b", empty_df(schema_7b_stream), batch_id)
                    update_monitor_from_batch("q6c", empty_df(schema_7c_files), batch_id)
                    return

                scored_batch = score_weather_blocks(weather_blocks)
                pred_batch = (
                    scored_batch
                      .select(*cols_7a)
                      .dropDuplicates(["record_key"])
                      .cache()
                )

                try:
                    update_monitor_from_batch("q6a", pred_batch, batch_id)
                    touched_day_keys = collect_partition_keys(pred_batch, ["site_id", "date"])

                    existing_q7a = read_active_table(Q7A_TABLE, schema_7a_files, touched_day_keys)
                    q7a_touched = dedupe_latest(
                        existing_q7a.unionByName(pred_batch, allowMissingColumns=True),
                        ["record_key"],
                        ["event_time"]
                    ).select(*cols_7a).cache()

                    try:
                        q7a_updates = write_partition_versions(q7a_touched, Q7A_TABLE, ["site_id", "date"], touched_day_keys, batch_id)
                        q7a_stage = stage_parquet_if_rows(pred_batch.select(*cols_7a), Q7A_TABLE, batch_id)
                    finally:
                        q7a_touched.unpersist()

                    building_batch = (
                        pred_batch
                          .select(
                              "record_key",
                              "wdw_start",
                              "wdw_end",
                              "date",
                              "hour_block",
                              "building_id",
                              "site_id",
                              F.col("prediction").alias("energy_consumption_6h"),
                              "event_time",
                              "weather_rows_in_block"
                          )
                          .dropDuplicates(["record_key"])
                          .cache()
                    )

                    try:
                        update_monitor_from_batch("q6b", building_batch, batch_id)
                        existing_q7b = read_active_table(Q7B_TABLE, schema_7b_stream, touched_day_keys)
                        q7b_touched = dedupe_latest(
                            existing_q7b.unionByName(building_batch, allowMissingColumns=True),
                            ["record_key"],
                            ["event_time"]
                        ).select(*cols_7b).cache()

                        try:
                            q7b_updates = write_partition_versions(q7b_touched, Q7B_TABLE, ["site_id", "date"], touched_day_keys, batch_id)
                            q7b_stage = stage_parquet_if_rows(building_batch.select(*cols_7b), Q7B_TABLE, batch_id)

                            touched_site_daily = build_site_daily_from_building_blocks(q7b_touched).select(*cols_7c).cache()
                            try:
                                q7c_updates = write_partition_versions(touched_site_daily, Q7C_TABLE, ["site_id", "date"], touched_day_keys, batch_id)
                                q7c_stage = stage_parquet_if_rows(touched_site_daily.select(*cols_7c), Q7C_TABLE, batch_id)
                                update_monitor_from_batch("q6c", touched_site_daily, batch_id)

                                commit_manifest_updates({
                                    WEATHER_TABLE: weather_updates,
                                    Q7A_TABLE: q7a_updates,
                                    Q7B_TABLE: q7b_updates,
                                    Q7C_TABLE: q7c_updates,
                                }, batch_id)

                                publish_staged_parquet_files(q7a_stage, P7A_OUTBOX)
                                publish_staged_parquet_files(q7b_stage, P7B_OUTBOX)
                                publish_staged_parquet_files(q7c_stage, P7C_OUTBOX)
                            finally:
                                touched_site_daily.unpersist()
                        finally:
                            q7b_touched.unpersist()
                    finally:
                        building_batch.unpersist()
                finally:
                    pred_batch.unpersist()
            finally:
                weather_blocks.unpersist()
        finally:
            weather_state.unpersist()
    finally:
        weather_batch.unpersist()


q7_outputs = (
    weather_prepared
      .writeStream
      .queryName("q7_prediction_outputs")
      .outputMode("append")
      .trigger(processingTime="7 seconds")
      .option("checkpointLocation", "checkpoints/a2b_q7_prediction_outputs")
      .foreachBatch(write_prediction_outputs)
      .start()
)


In [ ]:
# Separate q7b stream removed.
# q7_prediction_outputs writes q7b compacted snapshots and Kafka outbox records.
print("q7b is written by q7_prediction_outputs.")


In [ ]:
# Makes sure the manifest-backed q7b table exists and can be read - optional testing cell.
from pyspark.sql import functions as F

try:
    b6h = read_active_table(Q7B_TABLE, schema_7b_stream)
    print("row count:", b6h.count())
    print("\nschema:")
    b6h.printSchema()

    print("\nlatest 5 rows:")
    b6h.orderBy(F.col("event_time").desc_nulls_last()).show(5, truncate=False)

except Exception as e:
    print("read error:", e)
    print("Run the section 2.7 q7_prediction_outputs setup cell before this optional check.")


In [ ]:
# Separate q7c stream removed.
# q7_prediction_outputs rebuilds q7c from compacted q7b state, grouped by exact site_id/date.
print("q7c is written by q7_prediction_outputs.")


## 2.8 Republishing Parquet Outputs to Kafka

In [ ]:
# Stream block-level prediction Kafka outbox to Kafka.
# The outbox is append-only so file streaming remains stable while correctness snapshots are compacted.
import os

P7A_OUTBOX = globals().get("P7A_OUTBOX", "parquet/a2b/_kafka_outbox/7a_predictions")
os.makedirs(P7A_OUTBOX, exist_ok=True)

schema_7a_files = StructType([
    StructField("record_key", StringType(), True),
    StructField("hour_block", StringType(), True),
    StructField("prediction", DoubleType(), True),
    StructField("date", DateType(), True),
    StructField("site_id", StringType(), True),
    StructField("building_id", StringType(), True),
    StructField("wdw_start", TimestampType(), True),
    StructField("wdw_end", TimestampType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("weather_rows_in_block", LongType(), True)
])

predictions_stream = (
    spark.readStream
         .format("parquet")
         .schema(schema_7a_files)
         .load(P7A_OUTBOX)
)

predictions_to_kafka = (
    predictions_stream
      .withColumn("key", F.col("record_key").cast("string"))
      .withColumn("value", F.to_json(F.struct(*predictions_stream.columns)))
      .select(F.col("key").cast("string"), F.col("value").cast("string"))
      .writeStream
      .queryName("task8_predictions_7a_to_kafka")
      .format("kafka")
      .option("kafka.bootstrap.servers", KAFKA_SERVERS)
      .option("topic", "predictions_7a")
      .option("checkpointLocation", "checkpoints/task8_pred_7a_outbox")
      .outputMode("append")
      .start()
)


In [ ]:
# Stream building 6-hour Kafka outbox to Kafka.
import os

P7B_OUTBOX = globals().get("P7B_OUTBOX", "parquet/a2b/_kafka_outbox/7b_building_6h")
os.makedirs(P7B_OUTBOX, exist_ok=True)

schema_7b_stream = StructType([
    StructField("record_key", StringType(), True),
    StructField("wdw_start", TimestampType(), True),
    StructField("wdw_end", TimestampType(), True),
    StructField("date", DateType(), True),
    StructField("hour_block", StringType(), True),
    StructField("building_id", StringType(), True),
    StructField("site_id", StringType(), True),
    StructField("energy_consumption_6h", DoubleType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("weather_rows_in_block", LongType(), True)
])

bldg6h_stream = (
    spark.readStream
         .format("parquet")
         .schema(schema_7b_stream)
         .load(P7B_OUTBOX)
)

bldg6h_to_kafka = (
    bldg6h_stream
      .select(
          F.col("record_key").cast("string").alias("key"),
          F.to_json(F.struct(*[F.col(c) for c in bldg6h_stream.columns])).alias("value")
      )
      .selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")
      .writeStream
      .queryName("task8_building_6h_to_kafka")
      .format("kafka")
      .option("kafka.bootstrap.servers", KAFKA_SERVERS)
      .option("topic", "agg_building6h_7b")
      .option("checkpointLocation", "checkpoints/task8_bldg6h_7b_outbox")
      .outputMode("append")
      .start()
)


In [ ]:
#FOR TESTING - CHECK IF STREAM 2 IS READABLE (if we are able to read it through a simple consumer)
chk = (spark.readStream.format("kafka")
       .option("kafka.bootstrap.servers","kafka:9092")
       .option("subscribe","agg_building6h_7b")
       .option("startingOffsets","earliest")
       .load()
       .selectExpr("CAST(key AS STRING) AS key",
                   "CAST(value AS STRING) AS value"))

q_chk = (chk.writeStream.format("console")
         .outputMode("append")
         .option("truncate", False)
         .start())

In [29]:
q_chk.stop() #Stop test once shown that stream is readable in Console

In [ ]:
# Stream site daily Kafka outbox to Kafka.
import os
from pyspark.sql.types import ArrayType, BooleanType

P7C_OUTBOX = globals().get("P7C_OUTBOX", "parquet/a2b/_kafka_outbox/7c_site_daily")
os.makedirs(P7C_OUTBOX, exist_ok=True)

schema_7c_files = StructType([
    StructField("site_id", StringType(), True),
    StructField("date", DateType(), True),
    StructField("daily_consumption_site", DoubleType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("wdw_start", TimestampType(), True),
    StructField("wdw_end", TimestampType(), True),
    StructField("building_block_count", LongType(), True),
    StructField("hour_block_count", LongType(), True),
    StructField("observed_hour_blocks", ArrayType(StringType()), True),
    StructField("missing_hour_blocks", ArrayType(StringType()), True),
    StructField("has_all_hour_blocks", BooleanType(), True),
    StructField("site_building_count", LongType(), True),
    StructField("expected_block_count", LongType(), True),
    StructField("is_complete_day", BooleanType(), True),
    StructField("record_key", StringType(), True)
])

site_daily_stream = (
    spark.readStream
         .format("parquet")
         .schema(schema_7c_files)
         .load(P7C_OUTBOX)
)

site_daily_to_kafka = (
    site_daily_stream
      .select(
          F.col("record_key").cast("string").alias("key"),
          F.to_json(F.struct(*[F.col(c) for c in site_daily_stream.columns])).alias("value")
      )
      .selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")
      .writeStream
      .queryName("task8_site_daily_to_kafka")
      .format("kafka")
      .option("kafka.bootstrap.servers", KAFKA_SERVERS)
      .option("topic", "agg_site_daily_7c")
      .option("checkpointLocation", "checkpoints/task8_site_daily_7c_outbox")
      .outputMode("append")
      .start()
)


In [ ]:
#FOR TESTING - CHECK IF STREAM 3 IS READABLE 
chk_2 = (spark.readStream.format("kafka")
       .option("kafka.bootstrap.servers","kafka:9092")
       .option("subscribe","agg_site_daily_7c")
       .option("startingOffsets","earliest")
       .load()
       .selectExpr("CAST(key AS STRING) AS key",
                   "CAST(value AS STRING) AS value"))

q_chk_2 = (chk_2.writeStream.format("console")
         .outputMode("append")
         .option("truncate", False)
         .start())

In [57]:
q_chk_2.stop()

## Clean Restart Utilities

**Reset utilities**

Use the following cleanup cells only when intentionally starting a fresh replay. They stop active streams and remove local Parquet/checkpoint state so Spark does not resume from previous offsets.

In [25]:
#If it takes too long, run spark.stop() directly instead
for q in spark.streams.active:
    q.stop()

In [ ]:
# Remove streaming Parquet outputs and local compaction/version state for a clean run.

import shutil, os
for p in ["parquet/a2b/7a_predictions",
          "parquet/a2b/7b_building_6h",
          "parquet/a2b/7c_site_daily",
          "parquet/a2b/_state",
          "parquet/a2b/_manifest",
          "parquet/a2b/_versions",
          "parquet/a2b/_outbox_staging",
          "parquet/a2b/_kafka_outbox"]:
    if os.path.exists(p):
        shutil.rmtree(p, ignore_errors=True)


In [27]:
# TO DELETE ALL CHECKPOINTS (don't want to have different checkpoints conflicting)

import shutil 
import os
shutil.rmtree("checkpoints", ignore_errors=True)
os.makedirs("checkpoints", exist_ok=True)

In [28]:
spark.stop()